In [ ]:
# Step 1: Install Dependencies
!pip install torch torchvision torchaudio
!pip install opencv-python-headless
!pip install easyocr
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt

In [ ]:
!pwd

In [ ]:
import os
import cv2
import torch
import easyocr
import numpy as np
import matplotlib.pyplot as plt
from yolov5.models.common import DetectMultiBackend
from yolov5.utils.general import non_max_suppression
from yolov5.utils.torch_utils import select_device

model_path = "/content/models/best.pt"  # Update this path with your trained model
device = select_device('cuda' if torch.cuda.is_available() else 'cpu')
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Error: Model file {model_path} not found!")


#Load
model = DetectMultiBackend(model_path, device=device, dnn=False)
stride, names, pt = model.stride, model.names, model.pt

print("✅ Model loaded successfully!")


In [ ]:
def detect_and_crop_plate(image_path, output_dir="/content/cropped_plates", padding=10):

    # Load the input image
    original_image = cv2.imread(image_path)
    img_height, img_width = original_image.shape[:2]

    # Resize image to 640x640 (Required for YOLOv5)
    resized_image = cv2.resize(original_image, (640, 640))

    # Convert image to tensor
    img_tensor = torch.from_numpy(resized_image).float() / 255.0  # Normalize
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0).to(device)  # Change to (batch, C, H, W)

    # Run inference
    pred = model(img_tensor)

    # Apply Non-Maximum Suppression (NMS)
    detections = non_max_suppression(pred, conf_thres=0.3, iou_thres=0.4)[0]

    # Check if detections exist
    if detections is None or len(detections) == 0:
        print("❌ No license plates detected!")
        return None

    os.makedirs(output_dir, exist_ok=True)
    cropped_plate_paths = []

    for i, det in enumerate(detections):
        x1, y1, x2, y2, conf, cls = det.tolist()

        # Rescale coordinates to original image size
        x1, y1, x2, y2 = (x1 * img_width / 640, y1 * img_height / 640,
                          x2 * img_width / 640, y2 * img_height / 640)

        # Apply padding while ensuring coordinates stay within the image bounds
        x1 = max(0, int(x1) - padding)
        y1 = max(0, int(y1) - padding)
        x2 = min(img_width, int(x2) + padding)
        y2 = min(img_height, int(y2) + padding)

        # Crop the license plate region
        cropped_plate = original_image[y1:y2, x1:x2]

        # Save cropped plate
        plate_path = os.path.join(output_dir, f"cropped_plate_{i}.jpg")
        cv2.imwrite(plate_path, cropped_plate)
        cropped_plate_paths.append(plate_path)

        print(f"✅ License plate saved to: {plate_path}")

    return cropped_plate_paths


In [ ]:
def preprocess_and_ocr(image_path):

    # Load the cropped plate
    img = cv2.imread(image_path)

    # Convert to Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply Histogram Equalization (Enhances contrast)
    enhanced = cv2.equalizeHist(gray)

    # Save the grayscale image
    processed_path = image_path.replace(".jpg", "_gray.jpg")
    cv2.imwrite(processed_path, enhanced)

    # Display the grayscale image
    plt.imshow(enhanced, cmap='gray')
    plt.axis('off')
    plt.title("Grayscale License Plate")
    plt.show()

    print(f"✅ Grayscale plate saved to: {processed_path}")

    # Run EasyOCR
    reader = easyocr.Reader(['en'])
    result = reader.readtext(enhanced)

    # Extract and print detected text
    extracted_texts = [detection[1] for detection in result]
    print(f"📝 Extracted Text: {extracted_texts}")

    return extracted_texts


In [ ]:
# Input Car Image
car_image_path = "/content/50-swift-car-number-plate-frame-all-model-fit-yogiji-12-original-imag9dhhb9jz42xs.png"  # Replace with actual image

# Step 1: Detect & Crop License Plates
cropped_plate_paths = detect_and_crop_plate(car_image_path, padding=10)

# Step 2: Process Each Cropped Plate and Extract Text
if cropped_plate_paths:
    final_results = []
    for plate_path in cropped_plate_paths:
        text = preprocess_and_ocr(plate_path)
        final_results.append(text)

    print("🚀 Final Detected License Plates:", final_results)
else:
    print("❌ No license plates found.")
